# B1 – Opisna statistika

**Sklop A: ena spremenljivka · samostojno delo · Telekom**

Notebook spremlja razdelek **A. Ena spremenljivka**

Vsa potrebna koda je pripravljena. Izvajaj celice, preberi rezultat in nato v posamezni
vaji **zamenjaj predlagani stolpec**. Ne spreminjaj izvornih CSV-jev. Številčne rešitve
niso vnaprej zapisane; ugotovitve dopiši v označene besedilne celice.

**Najprej** naredi osnovni pregled in vaje 1–8. **Za samostojen preizkus** uporabi
zaključno nalogo 9 (približno 7 minut). Dodatni predlogi so namenjeni hitrejšim udeležencem.


## Priprava

Notebook je samostojen in deluje lokalno ter v Google Colabu. Celice izvajaj od
zgoraj navzdol s **Shift + Enter**. Ob menjavi stolpcev ponovi ustrezne naslednje celice.

Prva celica poišče `utils/orodja.py`; v Colabu po potrebi prenese repozitorij in namesti
knjižnice iz `requirements.txt`. Lokalno jih enkrat namestimo po navodilih v README.
Druga celica pokliče skupni `poskrbi_za_podatke()` in vrne mapo `DATA`.
Funkcija uporabi pripravljene ali že prenesene podatke; manjkajoče prenese s Kaggla.
CSV-je nato beremo iz `DATA`, ne glede na mapo, v kateri je zvezek odprt.
Izvornih vrednosti ne popravljamo in rezultatov vaj ne zapisujemo samodejno v datoteke.


In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


### Skupni podatkovni vhod
`poskrbi_za_podatke()` iz `utils/orodja.py` pripravi skupno mapo podatkov in jo vrne kot `DATA`.

V tem sklopu beremo `DATA / "telekom_izvirni.csv"`: vseh 100 stolpcev z izvirnimi imeni.
Uvod in vizualizacija uporabljata 22-stolpčni slovenski izbor `telekom.csv`;
obe tabeli izhajata iz istih strank in ju pripravi isti podatkovni vhod.


In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 15)
pd.set_option("display.max_colwidth", 35)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11})


In [ ]:
df = pd.read_csv(DATA / "telekom_izvirni.csv")
print(f"Vrstic: {df.shape[0]:,}; stolpcev: {df.shape[1]}")
display(df.head())


### Kaj predstavlja zapis in kaj pomenijo predlagani stolpci?

Ena vrstica predstavlja zapis stranke v tej zbirki. Točno časovno okno in merske
enote posameznih agregatov je treba preveriti v dokumentaciji; denarnih vrednosti
ne bomo samodejno imenovali evri. `Customer_ID` je identifikator, ne meritev.

| Stolpec | Kaj opazujemo | Primerne metode |
|---|---|---|
| `rev_Mean` | Povprečni mesečni prihodek / obračunani znesek v izvornih enotah | Sredina, razpršenost, IQR, histogram |
| `mou_Mean` | Povprečna mesečna uporaba v minutah | Sredina, razpršenost, histogram |
| `totmrc_Mean` | Povprečni mesečni ponavljajoči se obračun | Sredina, razpršenost, IQR |
| `roam_Mean` | Agregirana mera gostovanja | Ničle, histogram, pozitivne vrednosti |
| `custcare_Mean` | Agregirano povprečje stikov s podporo | Ničle, frekvence, histogram; ni nujno celo število |
| `months` | Trajanje razmerja v mesecih | Diskretne vrednosti, mediana, frekvence |
| `hnd_price` | Cena telefona z malo različnimi vrednostmi | Frekvence, modus, mediana |
| `crclscod`, `area` | Kreditni / regionalni kodi | Kardinalnost, frekvence, redke kategorije |
| `income` | Koda dohodkovnega razreda, ne denarni znesek | Frekvence; urejenost preverimo v slovarju |
| `eqpdays` | Mera starosti opreme v dnevih | Razpon, kvartili, nenavadne vrednosti |

Oznaka `churn` je binarna ciljna spremenljivka za kasnejše napovedovanje.
V tem notebooku opisujemo posamezne stolpce; še ne iščemo odnosov med njimi.


## 1. Prvi pregled z `describe()`

Začni s spodnjim izborom, nato dodaj `totmrc_Mean`, `custcare_Mean` ali `eqpdays`.
Primerjaj `count` s številom vrstic. Iz podatkovnega tipa še ne sklepaj, katera
statistika ima poslovni pomen.


In [ ]:
opazovani_stolpci = ["rev_Mean", "mou_Mean", "roam_Mean", "months", "hnd_price"]

display(df[opazovani_stolpci].dtypes.rename("tip_v_pandasu").to_frame())
display(df[opazovani_stolpci].describe().T.round(2))


**Preveri:** v katerem stolpcu se `count` razlikuje od števila vrstic? Kje je
maksimum daleč nad 75. percentilom? Zakaj povprečje `Customer_ID` ne bi bilo uporabno?


## 2. Središčnost: povprečje, mediana in modus

**Začni:** `rev_Mean`. **Ponovi:** `mou_Mean` ali `totmrc_Mean`.
**Dodaten primer:** `hnd_price`, kjer lahko modus odraža ponavljajoče se cene.

Napovej, ali bo povprečje višje od mediane. Pandas pri teh izračunih izpusti
prepoznane manjkajoče vrednosti. Modusov je lahko več, zato vrne serijo vrednosti.


In [ ]:
stolpec_sredina = "rev_Mean"  # Poskusi še "mou_Mean", "totmrc_Mean" ali "hnd_price".
s_sredina = df[stolpec_sredina]

display(s_sredina.agg(["count", "mean", "median", "min", "max"]).round(2))
modusi = s_sredina.mode()
print(f"Število modusov: {len(modusi)}")
display(modusi.head(10).rename("modus").to_frame())


**Zapiši:** kateri podatek bolje opiše običajno vrednost izbranega stolpca? Kaj bi
lahko povzročilo razliko med povprečjem in mediano? To domnevo preveri na histogramu.


## 3. Razpršenost: razpon, varianca, odklon, kvartili in IQR

**Začni:** `rev_Mean`. **Ponovi:** `mou_Mean`, `months` ali `eqpdays`.

Standardni odklon je v enotah stolpca, varianca v kvadratih teh enot. Pri različnih
enotah velikosti odklonov ne primerjamo neposredno. Uporabljamo Pandasovo vzorčno
varianco in odklon z **`ddof=1`** (deljenje z n − 1); za deljenje z n bi izbrali `ddof=0`.


In [ ]:
stolpec_razprsenost = "rev_Mean"  # Možnosti: "mou_Mean", "months", "eqpdays".
s_razprsenost = df[stolpec_razprsenost].dropna()
q1_r, q2_r, q3_r = s_razprsenost.quantile([0.25, 0.50, 0.75])

display(pd.Series({
    "znanih vrednosti": s_razprsenost.size,
    "minimum": s_razprsenost.min(),
    "maksimum": s_razprsenost.max(),
    "razpon": s_razprsenost.max() - s_razprsenost.min(),
    "varianca (ddof=1)": s_razprsenost.var(ddof=1),
    "standardni odklon (ddof=1)": s_razprsenost.std(ddof=1),
    "Q1": q1_r,
    "Q2 / mediana": q2_r,
    "Q3": q3_r,
    "IQR": q3_r - q1_r,
}).round(2).to_frame(stolpec_razprsenost))


**Preveri:** kateri del podatkov opisuje IQR? Kako bi ena zelo visoka vrednost
vplivala na razpon, standardni odklon in kvartile? Ali opaziš vrednost, katere
pomen bi moral preveriti pri skrbniku podatkov?


## 4. Oblika porazdelitve in ničle

**Začni:** `roam_Mean`. **Ponovi:** `custcare_Mean` ali `mou_Mean`.

Primerjaj vse znane vrednosti s pozitivnimi vrednostmi. Imenovalec pri deležu ničel
so **znane vrednosti**, zato najprej uporabimo `.dropna()`. Manjkajoč zapis ni isto
kot ničla. Desni histogram prikaže pozitivne vrednosti do njihovega 99. percentila,
da en zelo velik ekstrem ne zakrije glavnine. Omejitev velja le za prikaz; izpisi
povprečij še vedno uporabljajo vse znane oziroma vse pozitivne vrednosti.


In [ ]:
stolpec_oblika = "roam_Mean"  # Možnosti: "custcare_Mean", "mou_Mean".
s_oblika = df[stolpec_oblika].dropna()
s_pozitivni = s_oblika.loc[s_oblika.gt(0)]

display(pd.Series({
    "znanih vrednosti": s_oblika.size,
    "manjkajočih vrednosti": df[stolpec_oblika].isna().sum(),
    "delež ničel med znanimi (%)": s_oblika.eq(0).mean() * 100,
    "povprečje med znanimi": s_oblika.mean(),
    "mediana med znanimi": s_oblika.median(),
    "povprečje med pozitivnimi": s_pozitivni.mean(),
}).round(2).to_frame(stolpec_oblika))

fig, osi = plt.subplots(1, 2, figsize=(11, 4))
osi[0].hist(s_oblika, bins=50, edgecolor="white")
osi[0].set_title("Vse znane vrednosti, tudi ničle")
meja_pozitivnih = s_pozitivni.quantile(0.99)
pozitivni_za_prikaz = s_pozitivni.loc[s_pozitivni.le(meja_pozitivnih)]
osi[1].hist(pozitivni_za_prikaz, bins=40, edgecolor="white")
osi[1].set_title("Pozitivne vrednosti do 99. percentila")
for os in osi:
    os.set_xlabel(stolpec_oblika)
    os.set_ylabel("Število strank")
fig.tight_layout()
plt.show()


**Preveri:** ali mediana 0 pomeni, da nihče ne uporablja storitve? Kaj izgubiš,
če poročaš le povprečje? Poskusi 20 in 80 razredov histograma ter primerjaj videz.
Izoliranje pozitivnih vrednosti je tukaj opis podskupine, ne čiščenje celotne tabele.


## 5. Dtype ni isto kot pomen: diskretne vrednosti in kode

**Začni:** `hnd_price`. **Primerjaj:** `months` in `income`.

Cena telefona ostaja numerična veličina tudi, če se ponavlja malo cen; uporabna je
tudi frekvenčna tabela. `income` je koda razreda, ne znesek. Pred uporabo mediane
ali urejanjem kod preveri, ali dokumentacija res potrjuje njihov vrstni red.


In [ ]:
for stolpec_tip in ["hnd_price", "months", "income"]:
    print()
    print(f"{stolpec_tip}: dtype={df[stolpec_tip].dtype}; "
          f"različnih znanih vrednosti={df[stolpec_tip].nunique()}")
    display(df[stolpec_tip].value_counts(dropna=False).sort_index().rename("število").to_frame())


**Odgovori:** kateri od teh stolpcev je število, kateri cena in kateri koda?
Zakaj `float64` sam po sebi ne pove, ali smemo računati povprečje?


## 6. Kategorije: frekvence, deleži in redke kategorije

**Začni:** `crclscod`. **Ponovi:** `area`, `new_cell` ali `hnd_webcap`.

`value_counts(normalize=True)` vrne deleže. Z `dropna=False` med kategorijami
prikažemo tudi prepoznane manjkajoče vrednosti; deleži se zato nanašajo na vse vrstice.
`nunique()` jih privzeto ne šteje. Prag redkosti lahko spremeniš.


In [ ]:
stolpec_kategorije = "crclscod"  # Možnosti: "area", "new_cell", "hnd_webcap".
prag_redkosti = 100
s_kategorije = df[stolpec_kategorije]

stevila_k = s_kategorije.value_counts(dropna=False)
delezi_k = s_kategorije.value_counts(dropna=False, normalize=True) * 100
frekvence_k = pd.DataFrame({"število": stevila_k, "delež (%)": delezi_k})
redke_k = stevila_k.loc[stevila_k.lt(prag_redkosti)]

print(f"Različnih znanih kategorij: {s_kategorije.nunique()}")
print(f"Kategorij z manj kot {prag_redkosti} zapisi: {len(redke_k)}")
print(f"Pet najpogostejših kategorij pokrije {delezi_k.head(5).sum():.2f} % zapisov.")
display(frekvence_k.round(2))
display(redke_k.rename("število v redki kategoriji").to_frame())


In [ ]:
# Pokažemo do 15 najpogostejših kategorij. Ostale ostanejo v podatkih.
prikaz_k = s_kategorije.fillna("MANJKA").value_counts().head(15)
fig, os = plt.subplots(figsize=(10, 4))
prikaz_k.plot.bar(ax=os)
os.set(xlabel=stolpec_kategorije, ylabel="Število strank",
       title="Do 15 najpogostejših kategorij")
fig.tight_layout()
plt.show()


**Preveri:** ali prvih pet kategorij opiše večino strank? Zakaj sklepa na osnovi
zelo redke kategorije ne bi posplošil? Ali graf prikazuje vse kategorije?


## 7. Manjkajoče vrednosti in posebne oznake

Izračunaj število in delež manjkajočih vrednosti. Nato samo **preštej**, koliko
vrstic bi ostalo pri splošnem `dropna()`; vrstic ne briši.

Posebne kode, kot sta `U` ali `UNKW`, obravnavamo ločeno od `NaN`. Njihov pomen je
treba preveriti za vsak stolpec. Ne pretvarjaj vseh črk `U` v manjkajoče vrednosti.


In [ ]:
manjkajoci = pd.DataFrame({
    "znanih": df.count(),
    "manjkajočih": df.isna().sum(),
    "manjka (%)": df.isna().mean() * 100,
}).sort_values("manjka (%)", ascending=False)

display(manjkajoci.head(15).round(2))
print(f"Stolpcev z vsaj eno manjkajočo vrednostjo: {df.isna().any().sum()}")
print(f"Vrstic z vsaj eno manjkajočo vrednostjo: {df.isna().any(axis=1).sum():,}")
print(f"Po splošnem dropna() bi ostalo {df.notna().all(axis=1).sum():,} vrstic.")
print(f"To je {df.notna().all(axis=1).mean() * 100:.2f} % začetnih podatkov.")


In [ ]:
stolpec_oznake = "marital"  # Poskusi še "new_cell" ali "hnd_webcap".
display(df[stolpec_oznake].value_counts(dropna=False).rename("število").to_frame())


**Zapiši:** ali bi zaradi manjkajočega podatka v kateremkoli od 100 stolpcev
izbrisal celotno stranko? Kaj pri prikazu `marital` pove `NaN` in česa brez
dokumentacije še ne vemo o kodi `U`? Zakaj je prikaz imenovalca pomemben?
